# Area of Habitat (AOH) – Lanius meridionalis

Map of species potential range: **where it actually occurs** (GBIF) vs **Area of Habitat** (ML model).

## Area of Habitat – methodology (research)

**AOH** (Area of Habitat) is an IUCN/BirdLife concept: "habitat available for the species, i.e. habitat within its range". It complements geographic range maps by showing potential presence and reducing commission errors (false positives).

**Classical AOH method (IUCN/BirdLife):**
- Deductive: subtracting unsuitable areas from species range
- Uses: habitat preferences (IUCN Habitat Classification), elevation limits (SRTM)
- Translation table: IUCN Habitat → land cover (Copernicus CGLS-LC100)
- Resolution ~100 m

**Our approach (ML model):**
- Inductive: XGBoost model learns from GBIF data + terrain features (elevation, slope, land cover %, OSM)
- **Observed**: hexes with GBIF observations (ground truth)
- **Area of Habitat**: hexes where model predicts presence (p > threshold) – potential habitat based on environmental features
- Toggle layer (LayerControl) as in showcase map

**Sources:** Brooks et al. (2019) Trends Ecol Evol; Lumbierres et al. Nature Scientific Data (2022); BirdLife code_for_AOH.

In [ ]:
%pip install -q pandas numpy scikit-learn h3 pyarrow boto3 folium xgboost

In [4]:
# Config (aligned with 02_species_showcase_map)
COUNTRY = "ES"
H3_RES = 7
OUTPUT_DIR = "output"
XGB_MODEL_PATH = f"{OUTPUT_DIR}/species_predictor_xgb.pkl"
TARGET_SPECIES_PATH = f"{OUTPUT_DIR}/target_species.csv"
SHOWCASE_PATH = f"{OUTPUT_DIR}/showcase_data.pkl"
USE_XGBOOST = True

# Species for AOH map (species_id or species_name)
SPECIES_FOR_AOH = "Lanius meridionalis"

# FULL_SPAIN=True: fetch all ES res-7 hexes, replicate feature engineering, run prediction on all Spain
# FULL_SPAIN=False: use showcase_data (subset of hexes from test set only)
FULL_SPAIN = True

# When FULL_SPAIN: AOH threshold (0.5 = conservative)
AOH_PROB_THRESHOLD_FULL = 0.5

# When FULL_SPAIN=False: thresholds as toggle layers
AOH_PROB_THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5]

# S3 (for FULL_SPAIN)
S3_BUCKET = "ie-datalake"
GOLD_GEE_TERRAIN = "gold/gee_hex_terrain"
GOLD_OSM_HEX = "gold/osm_hex_features"
GOLD_NATURE2000 = "gold/nature2000_cell_protection"
GOLD_CELL_METRICS = "gold/gbif_cell_metrics"
GOLD_H3_MAPPING = "gold/gbif_species_h3_mapping"
GEE_TERRAIN_SNAPSHOT = "2019"
NATURE2000_SNAPSHOT_DATE = "2026-02-27"
AWS_PROFILE = "486717354268_PowerUserAccess"

## 1) Load data and model (FULL_SPAIN or showcase_data)

In [6]:
import pickle
import sys
from pathlib import Path

import h3
import numpy as np
import pandas as pd

cwd = Path.cwd()
sys.path.insert(0, str(cwd))
if (cwd / "gbif_species_predictor").is_dir():
    sys.path.insert(0, str(cwd / "gbif_species_predictor"))

# Load XGB model (has scaler + feat_cols)
xgb_path = Path(XGB_MODEL_PATH)
if not xgb_path.exists():
    xgb_path = Path("gbif_species_predictor") / XGB_MODEL_PATH
with open(xgb_path, "rb") as f:
    xgb_obj = pickle.load(f)

scaler = xgb_obj["scaler"]
feat_cols = xgb_obj.get("feature_cols", [])
xgb_multi = xgb_obj["model"]

# Target species from CSV
target_path = Path(TARGET_SPECIES_PATH)
if not target_path.exists():
    target_path = Path("gbif_species_predictor") / TARGET_SPECIES_PATH
df_target_species = pd.read_csv(target_path)
target_species_ids = df_target_species["species_id"].astype(int).tolist()
species_names = dict(zip(df_target_species["species_id"].astype(int), df_target_species["species_name"]))
h3_col = "h3_index"

if FULL_SPAIN:
    # ─── FULL_SPAIN: load all hexes from S3, build features, predict ───
    import boto3
    import pyarrow.dataset as ds
    import pyarrow.fs as pafs

    session = boto3.Session(profile_name=AWS_PROFILE)
    creds = session.get_credentials().get_frozen_credentials()
    region = session.region_name or "eu-west-2"
    fs_read = pafs.S3FileSystem(access_key=creds.access_key, secret_key=creds.secret_key, session_token=creds.token, region=region)

    def _read_parquet(path: str) -> pd.DataFrame:
        tbl = ds.dataset(path, filesystem=fs_read, format="parquet").scanner().to_table()
        return tbl.to_pandas()

    # 1) All hexes from GEE terrain
    path_gee = f"{S3_BUCKET}/{GOLD_GEE_TERRAIN}/country={COUNTRY}/snapshot={GEE_TERRAIN_SNAPSHOT}/h3_resolution={H3_RES}"
    df_gee = _read_parquet(path_gee)
    gee_h3 = "h3_index" if "h3_index" in df_gee.columns else "h3_id"
    df_hexes = df_gee[[gee_h3]].drop_duplicates().rename(columns={gee_h3: h3_col})

    # 2) Build features (OSM, GEE, Nature2000, cell metrics) – aligned with 01_species_predictor
    from data_loader import _rename_lc_columns

    df = df_hexes.copy()

    # OSM
    path_osm = f"{S3_BUCKET}/{GOLD_OSM_HEX}/country={COUNTRY}/h3_resolution={H3_RES}"
    df_osm = _read_parquet(path_osm)
    rc = "h3_index" if "h3_index" in df_osm.columns else "h3_id"
    df = df.merge(df_osm, left_on=h3_col, right_on=rc, how="left")
    if rc != h3_col and rc in df.columns:
        df = df.drop(columns=[rc])

    # Nature2000
    path_n2k = f"{S3_BUCKET}/{GOLD_NATURE2000}/country={COUNTRY}/h3_resolution={H3_RES}/snapshot_date={NATURE2000_SNAPSHOT_DATE}"
    try:
        df_n2k = _read_parquet(path_n2k)
        nc = "h3_index" if "h3_index" in df_n2k.columns else "h3_id"
        n2k = df_n2k[[nc, "is_protected_area", "nearest_protected_distance"]].rename(columns={nc: h3_col})
        n2k["is_protected_area"] = (n2k["is_protected_area"].astype(str).str.lower() == "yes").astype(np.float32)
        n2k["nearest_protected_distance"] = pd.to_numeric(n2k["nearest_protected_distance"], errors="coerce").fillna(-1).astype(np.float32)
        df = df.merge(n2k, on=h3_col, how="left")
    except Exception as e:
        print(f"Skip Nature2000: {e}")

    # GEE terrain (exclude country, snapshot)
    gee_cols = [c for c in df_gee.columns if c not in (gee_h3, "h3_id", "h3_resolution", "country", "snapshot")]
    df_gee_sub = df_gee[[gee_h3] + gee_cols].rename(columns={gee_h3: h3_col})
    df = df.merge(df_gee_sub, on=h3_col, how="left")
    df = _rename_lc_columns(df)

    # Cell metrics (2019–2023)
    try:
        parts = []
        for year in range(2019, 2024):
            path_cell = f"{S3_BUCKET}/{GOLD_CELL_METRICS}/country={COUNTRY}/year={year}/h3_resolution={H3_RES}"
            try:
                parts.append(_read_parquet(path_cell))
            except Exception:
                pass
        df_cell = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
        if not df_cell.empty:
            obs_col = "observation_count" if "observation_count" in df_cell.columns else "obs_count_all"
            cc = "h3_index" if "h3_index" in df_cell.columns else "h3_id"
            agg = df_cell.groupby(cc).agg({obs_col: "sum", **({"dqi": "mean"} if "dqi" in df_cell.columns else {})}).reset_index().rename(columns={cc: h3_col})
            if obs_col in agg.columns:
                agg["log_obs_count"] = np.log1p(agg[obs_col].fillna(0))
            df = df.merge(agg, on=h3_col, how="left")
            obs_map = dict(zip(df[h3_col], df["log_obs_count"].fillna(0)))
            def neighbor_mean(h, m):
                try:
                    nbs = [n for n in h3.grid_disk(h, 1) if n != h]
                    return np.mean([m.get(n, 0) for n in nbs]) if nbs else 0.0
                except Exception:
                    return 0.0
            df["neighbor_log_obs_mean"] = df[h3_col].map(lambda h: neighbor_mean(h, obs_map))
    except Exception as e:
        print(f"Skip cell metrics: {e}")

    df_feat = df.copy()

    # Add missing feat_cols as 0
    for c in feat_cols:
        if c not in df_feat.columns:
            df_feat[c] = 0.0

    X = df_feat[feat_cols].fillna(0).replace([np.inf, -np.inf], 0)
    X = np.clip(X.values.astype(np.float64), -1e15, 1e15).astype(np.float32)
    X = scaler.transform(X)

    BATCH_SIZE = 4096
    probs_list = []
    for start in range(0, len(X), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(X))
        probs_list.append(np.column_stack([p[:, 1] for p in xgb_multi.predict_proba(X[start:end])]).astype(np.float32))
    probs_test = np.vstack(probs_list)

    hex_test = df_feat[h3_col].values
    Y_test = None  # observed loaded from GOLD_H3_MAPPING in next cell
    test_mask = np.ones(len(df_feat), dtype=bool)

    print(f"FULL_SPAIN: {len(hex_test)} hexes, {len(target_species_ids)} species, prediction on all Spain")
else:
    # ─── Showcase mode: subset from test set ───
    showcase_path = Path(SHOWCASE_PATH)
    if not showcase_path.exists():
        showcase_path = Path("gbif_species_predictor") / SHOWCASE_PATH
    if not showcase_path.exists():
        raise FileNotFoundError(f"Run 01_species_predictor.ipynb and 02_species_showcase_map.ipynb first. Need {SHOWCASE_PATH}")

    with open(showcase_path, "rb") as f:
        show = pickle.load(f)

    df_feat = show["df_feat"]
    hex_test = show["hex_test"]
    Y_test = show["Y_test"]
    target_species_ids = show["target_species_ids"]
    species_names = show.get("species_names", {}) or species_names
    h3_col = show["h3_col"]
    test_mask = df_feat[h3_col].isin(hex_test).values

    for c in feat_cols:
        if c not in df_feat.columns:
            df_feat[c] = 0.0

    X_test = scaler.transform(df_feat.loc[test_mask, feat_cols].fillna(0).replace([np.inf, -np.inf], 0).values.astype(np.float32))
    probs_test = np.column_stack([p[:, 1] for p in xgb_multi.predict_proba(X_test)]).astype(np.float32)

    print(f"Loaded: {len(hex_test)} hexes, {len(target_species_ids)} species")

FULL_SPAIN: 95481 hexes, 10 species, prediction on all Spain


## 2) Select species and build hex sets

In [8]:
# Build id->name (df_target_species from load cell)
# Prefer proper names over numeric strings when CSV has duplicate species_id (e.g. 7341500 -> "Lanius meridionalis" not "7341500")
df_target = df_target_species
id_to_name = {}
for _, row in df_target.iterrows():
    sid = int(row["species_id"])
    name = str(row["species_name"]).strip()
    if sid not in id_to_name:
        id_to_name[sid] = name
    elif name != str(sid):  # overwrite numeric placeholder with proper name
        id_to_name[sid] = name
# Merge species_names but never overwrite proper name with numeric string (CSV may have duplicate ids)
for sid, name in species_names.items():
    sid = int(sid)
    if sid not in id_to_name:
        id_to_name[sid] = name
    elif name != str(sid) and id_to_name[sid] == str(sid):
        id_to_name[sid] = name  # upgrade numeric -> proper

sel = str(SPECIES_FOR_AOH).strip()
j = None
species_id = None
species_name = None
for idx, sid in enumerate(target_species_ids):
    sid_int = int(sid)
    name = id_to_name.get(sid_int, str(sid_int))
    if name == sel or str(sid_int) == sel:
        j = idx
        species_id = sid_int
        species_name = name
        break
if j is None:
    avail = [id_to_name.get(int(s), str(s)) for s in target_species_ids]
    raise ValueError(f"SPECIES_FOR_AOH='{sel}' not in model targets. Available: {avail}")

# Observed: GBIF ground truth
if FULL_SPAIN:
    import boto3
    import pyarrow.dataset as ds
    import pyarrow.fs as pafs
    PRESENCE_THRESHOLD = 2  # aligned with data_loader / species_predictor
    session = boto3.Session(profile_name=AWS_PROFILE)
    creds = session.get_credentials().get_frozen_credentials()
    fs_read = pafs.S3FileSystem(access_key=creds.access_key, secret_key=creds.secret_key, session_token=creds.token, region=session.region_name or "eu-west-2")
    def _read_pq(p):
        return ds.dataset(p, filesystem=fs_read, format="parquet").scanner().to_table().to_pandas()
    parts = []
    for year in range(2018, 2025):
        try:
            p = f"{S3_BUCKET}/{GOLD_H3_MAPPING}/country={COUNTRY}/year={year}/h3_resolution={H3_RES}"
            df_h = _read_pq(p)
            sid_col = "taxon_key" if "taxon_key" in df_h.columns else "species_id"
            # taxon_key may be string in parquet – compare as int
            df_h[sid_col] = pd.to_numeric(df_h[sid_col], errors="coerce")
            df_h = df_h[df_h[sid_col] == species_id]
            occ_col = "occurrence_count" if "occurrence_count" in df_h.columns else "obs_count_all"
            hc = "h3_index" if "h3_index" in df_h.columns else "h3_id"
            if occ_col in df_h.columns:
                df_h[occ_col] = pd.to_numeric(df_h[occ_col], errors="coerce").fillna(0)
                df_h = df_h[df_h[occ_col] >= PRESENCE_THRESHOLD]
            parts.append(df_h[[hc]])
        except Exception as e:
            pass  # partition may not exist
    observed_hexes = set()
    if parts:
        obs_df = pd.concat(parts, ignore_index=True).drop_duplicates()
        hc = "h3_index" if "h3_index" in obs_df.columns else "h3_id"
        observed_hexes = set(obs_df[hc].dropna().astype(str))
else:
    observed_hexes = set(hex_test[Y_test[:, j] > 0])

# Area of Habitat
if FULL_SPAIN:
    aoh_per_threshold = {AOH_PROB_THRESHOLD_FULL: set(hex_test[probs_test[:, j] >= AOH_PROB_THRESHOLD_FULL])}
else:
    aoh_per_threshold = {t: set(hex_test[probs_test[:, j] >= t]) for t in AOH_PROB_THRESHOLDS}

n_obs = len(observed_hexes)
print(f"{species_name}")
print(f"  Observed (GBIF): {n_obs} hexes")
for t in sorted(aoh_per_threshold.keys()):
    n = len(aoh_per_threshold[t])
    ov = len(observed_hexes & aoh_per_threshold[t])
    print(f"  AOH p≥{t}: {n} hexes, overlap {ov}")

ValueError: SPECIES_FOR_AOH='Lanius meridionalis' not in model targets. Available: ['7341500', 'Aythya ferina', 'Streptopelia turtur', 'Ichthyaetus audouinii', 'Neophron percnopterus', 'Pluvialis squatarola', 'Otis tarda', 'Aquila adalberti', 'Oxyura leucocephala', '7341500']

## 3) Folium map: Observed vs Area of Habitat (toggle layers)

In [ ]:
import folium

def h3_to_folium_coords(h):
    b = h3.cell_to_boundary(h)
    coords = [[p[0], p[1]] for p in b]
    if coords[0] != coords[-1]:
        coords.append(coords[0])
    return coords

m = folium.Map(location=[40.4, -3.7], zoom_start=6, tiles="CartoDB positron")

# Layer 1: Observed (GBIF)
fg_observed = folium.FeatureGroup(name=f"Observed (GBIF) – {n_obs}", show=True)
for h in observed_hexes:
    try:
        folium.Polygon(locations=h3_to_folium_coords(h), color="#3b82f6", weight=1, fill=True, fill_color="#3b82f6", fill_opacity=0.7, popup="Observed").add_to(fg_observed)
    except Exception:
        pass

# Layer 2: Area of Habitat (model)
fg_observed.add_to(m)
aoh_thresholds = sorted(aoh_per_threshold.keys())
default_thresh = AOH_PROB_THRESHOLD_FULL if FULL_SPAIN else AOH_PROB_THRESHOLDS[len(AOH_PROB_THRESHOLDS) // 2]
for t in aoh_thresholds:
    hexes_t = aoh_per_threshold[t]
    fg = folium.FeatureGroup(name=f"AOH p≥{t} – {len(hexes_t)}", show=(t == default_thresh))
    for h in hexes_t:
        try:
            folium.Polygon(locations=h3_to_folium_coords(h), color="#22c55e", weight=1, fill=True, fill_color="#22c55e", fill_opacity=0.5, popup=f"AOH p≥{t}").add_to(fg)
        except Exception:
            pass
    fg.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

# Legend
legend_lines = "<br>".join([f"AOH p≥{t}: {len(aoh_per_threshold[t])}" for t in aoh_thresholds])
legend_html = f"""
<div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; background: white; padding: 10px; border-radius: 5px; box-shadow: 0 2px 6px rgba(0,0,0,0.3);">
  <p><b>{species_name}</b></p>
  <p style="color:#3b82f6">● Observed (GBIF): {n_obs}</p>
  <p style="color:#22c55e">● AOH – select threshold in LayerControl:</p>
  <p style="font-size:0.9em">{legend_lines}</p>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
out_path = f"{OUTPUT_DIR}/area_of_habitat_{species_id}.html"
m.save(out_path)
print(f"Saved: {out_path}")

Saved: output/area_of_habitat_7341500.html


KeyboardInterrupt: 